In [7]:
import starfile
import os
from pathlib import Path
import pandas as pd

In [8]:
def split_mrcs(tomo_name: str):
    """
    Split an MRC file into individual micrographs.
    """
    import mrcfile
    import os
    from pathlib import Path
    from findingampa.utils.imod  import get_exclude_list

    mrcs_file = Path(f"{tomo_name}.mrc")  # Replace with your MRCs file path
    output_dir = Path("relion_tilts/")
    output_dir.mkdir(parents=True, exist_ok=True)

    exclude_list = get_exclude_list()
    exclude_list = [int(i) for i in exclude_list.split(",") if i.isdigit()]
    images = []

    with mrcfile.open(mrcs_file, mode='r') as mrc:
       
        for i in range(mrc.header["nz"]):
            if (i+1) in exclude_list:
                continue
            
            output_path = os.path.join(output_dir, f'{tomo_name}_{i}.mrc')
            if Path(output_path).exists():
                print(f"File {output_path} already exists. Skipping.")
                images.append((i,output_path))
                continue
            img = mrc.data[i]
            with mrcfile.new(output_path, overwrite=True) as new_mrc:
                new_mrc.set_data(img)
                new_mrc.update_header_from_data()  # Update header information
            images.append((i,output_path))
    return images

In [9]:
def create_relion_tomo_starfile():
    import starfile
    import pymeshlab
    import numpy as np
    import pandas as pd
    from yet_another_imod_wrapper.utils.xf import XF
    from yet_another_imod_wrapper.utils.io import read_tlt
    from yet_another_imod_wrapper.utils.etomo import get_input_tilt_axis_rotation_angle
    from findingampa.utils.imod import get_tomopitch_angle_offset
    from pathlib import Path
    try:
        tomo_name = list(Path.cwd().glob("*.rawtlt"))[0].stem
    except IndexError:
        typer.echo("No rawtlt file found")
        raise typer.Exit()
    
    image_list = split_mrcs(tomo_name)

    pre_exposure_data = pd.read_csv(f"../{tomo_name}.dose",sep="\s+",header=None)
    ctf_data = pd.read_csv(f"{tomo_name}.defocus",sep="\s+",header=None,skiprows=1)
    # COnvert firt column of ctf_data to 0-based index
    ctf_data.iloc[:, 0] = ctf_data.iloc[:, 0].astype(int) - 1
    ctf_data.index = ctf_data.iloc[:, 0]
    in_plane_rotation = get_input_tilt_axis_rotation_angle(
        "align.log"
    )
    xf = XF.from_file(f"{tomo_name}.xf", initial_tilt_axis_rotation_angle=in_plane_rotation)
    tilt_angles = read_tlt(f"{tomo_name}.tlt")
    align_angle_offset = get_tomopitch_angle_offset()
    if align_angle_offset is not None:
        tilt_angles += align_angle_offset
    shifts_angstom = xf.specimen_shifts * 2.5
    star_data = []

    for i, image in image_list:
        # Create a new star file
        star_data_row = {
            'rlnMicrographName': image,
            'rlnMicrographPreExposure': pre_exposure_data.iloc[i][0],
            'rlnDefocusU': ctf_data.loc[i][4],
            'rlnDefocusV': ctf_data.loc[i][5],
            'rlnDefocusAngle': ctf_data.loc[i][6],
            'rlnTomoXTilt': 0.0,
            'rlnTomoYTilt': tilt_angles[i],
            'rlnTomoZRot': xf.in_plane_rotations[i],
            'rlnTomoXShiftAngst': shifts_angstom[i][0],
            'rlnTomoYShiftAngst': shifts_angstom[i][1],
        }
        star_data.append(star_data_row)
        
    starfile.write({f"{tomo_name}": pd.DataFrame(star_data)}, f"{tomo_name}_relion_tiltseries.star", overwrite=True)

    tomo_set_data = [{
        'rlnTomoName': tomo_name,
        'rlnVoltage': 300,
        'rlnSphericalAberration': 2.7,

        'rlnAmplitudeContrast': 0.07, 
        'rlnMicrographOriginalPixelSize': 2.5, 
        'rlnTomoHand': -1,
        'rlnOpticsGroupName': '1', 
        'rlnTomoTiltSeriesPixelSize': 2.5,
        'rlnTomoTiltSeriesStarFile' : f"{tomo_name}_relion_tiltseries.star" 
    }]

    starfile.write({"global": pd.DataFrame(tomo_set_data)}, f"{tomo_name}_relion_tomoset.star", overwrite=True)



    

In [10]:
with open("/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS/tomo_for_sta.txt","r") as f:
    tomos_for_sta = f.readlines()

tomos_for_sta = ["20240209_waffleHip_165"]
for tomo in tomos_for_sta:
    tomo = tomo.strip()
    print(tomo)
    os.chdir(f'/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS/{tomo}/best_alignment/')
    create_relion_tomo_starfile()

20240209_waffleHip_165
File relion_tilts/20240209_waffleHip_165_3.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_4.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_5.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_6.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_7.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_8.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_9.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_10.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_11.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_12.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_13.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_14.mrc already exists. Skipping.
File relion_tilts/20240209_waffleHip_165_15.mrc already exists. Skipping.
File relion_tilts/2024

In [16]:
os.chdir("/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/analyses/relion5_2025/AlignTiltSeries/initial")
star_data = []
for tomo_star in Path("/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS/").glob("*/best_alignment/*_relion_tomoset.star"):
    tomo_data = starfile.read(tomo_star)
    tomo_data['rlnTomoTiltSeriesStarFile'] = f"AlignTiltSeries/initial/tilt_series/{tomo_data.iloc[0]['rlnTomoName']}_relion_tiltseries.star" 
    star_data.append(tomo_data)
star_data = pd.concat(star_data, ignore_index=True)
starfile.write({"global": star_data}, "relion_tomoset.star", overwrite=True)

In [76]:
os.chdir('/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS/20231026_HippAu_14/best_alignment')
create_relion_tomo_starfile()

21.68


In [26]:
os.chdir('/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS/20240111_WaffleHipp_227/best_alignment')
create_relion_tomo_starfile()

In [2]:
example = starfile.read("/home/gouauxlab/andecath/relion5_tutorial/Relion-5.0/AlignTiltSeries/job005/tilt_series/TS_01.star")

In [36]:
example.columns

Index(['rlnMicrographMovieName', 'rlnTomoTiltMovieFrameCount',
       'rlnTomoNominalStageTiltAngle', 'rlnTomoNominalTiltAxisAngle',
       'rlnMicrographPreExposure', 'rlnTomoNominalDefocus',
       'rlnCtfPowerSpectrum', 'rlnMicrographNameEven', 'rlnMicrographNameOdd',
       'rlnMicrographName', 'rlnMicrographMetadata', 'rlnAccumMotionTotal',
       'rlnAccumMotionEarly', 'rlnAccumMotionLate', 'rlnCtfImage',
       'rlnDefocusU', 'rlnDefocusV', 'rlnCtfAstigmatism', 'rlnDefocusAngle',
       'rlnCtfFigureOfMerit', 'rlnCtfMaxResolution', 'rlnCtfIceRingDensity',
       'rlnTomoXTilt', 'rlnTomoYTilt', 'rlnTomoZRot', 'rlnTomoXShiftAngst',
       'rlnTomoYShiftAngst'],
      dtype='object')

In [56]:
example_tomo_set = starfile.read("/home/gouauxlab/andecath/relion5_tutorial/Relion-5.0/AlignTiltSeries/job005/aligned_tilt_series.star")

In [58]:
example_tomo_set.columns

Index(['rlnTomoName', 'rlnVoltage', 'rlnSphericalAberration',
       'rlnAmplitudeContrast', 'rlnMicrographOriginalPixelSize', 'rlnTomoHand',
       'rlnOpticsGroupName', 'rlnTomoTiltSeriesPixelSize',
       'rlnTomoTiltSeriesStarFile', 'rlnEtomoDirectiveFile'],
      dtype='object')